# Copilot Phase 2 — Risk Segmentation & Early Warning

**Goal:** Use the competition model's predicted default probabilities + Phase 0 
monthly snapshots to rank every loan by risk level, profile each tier, 
and surface the loans that most urgently need intervention.

| Step | Output |
|---|---|
| 1 · Risk Band assignment | `risk_band` column (Low / Medium / High / Critical) |
| 2 · Risk Profile | `risk_profile.csv` — per-band summary statistics |
| 3 · Early Warning | deteriorating loans that were originally Low/Medium risk |
| 4 · Intervention Priority | `priority_loans.csv` — top-50 by priority score |

## 0 · Imports & Config

In [69]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
DATA_DIR = Path('.')

# Risk band thresholds
BAND_BINS   = [0, 0.10, 0.30, 0.60, 1.001]
BAND_LABELS = ['Low', 'Medium', 'High', 'Critical']

# Deterioration threshold (must match Phase 0)
DETERIORATION_THRESHOLD = -0.10

print('Ready.')

Ready.


## 1 · Load Raw Tables

In [70]:
loans           = pd.read_csv(DATA_DIR / 'loans.csv')
borrowers       = pd.read_csv(DATA_DIR / 'borrowers.csv')
payment_history = pd.read_csv(DATA_DIR / 'payment_history.csv')
train_labels    = pd.read_csv(DATA_DIR / 'train.csv')   # loan_id, days_to_early_default, default_flag

print(f'loans           : {loans.shape[0]:,} rows')
print(f'borrowers       : {borrowers.shape[0]:,} rows')
print(f'payment_history : {payment_history.shape[0]:,} rows')
print(f'train_labels    : {train_labels.shape[0]:,} rows  | defaults: {train_labels.default_flag.sum():,}')

loans           : 181,800 rows
borrowers       : 71,681 rows
payment_history : 627,511 rows
train_labels    : 125,999 rows  | defaults: 22,613


## 2 · Feature Engineering

Mirrors `Loan.ipynb` exactly — same feature set, same imputation rules — so the 
risk scores are consistent with the competition submission.

### 2.1 Payment-history aggregates (per loan)

In [71]:
# Sort chronologically so the expanding window is correct
ph = payment_history.sort_values(['loan_id', 'month_number']).copy()
ph['days_delinquent_filled'] = ph['days_delinquent'].fillna(0.0)
ph['on_time_flag']           = (ph['days_delinquent_filled'] == 0).astype(int)

# Aggregate to one row per loan: overall on_time_rate + cumulative delinquent days
ph_agg = (
    ph.groupby('loan_id')
    .agg(
        on_time_rate          = ('on_time_flag',           'mean'),
        total_days_delinquent = ('days_delinquent_filled', 'sum'),
        has_payment_history   = ('loan_id',                lambda x: 1),
    )
    .reset_index()
)

print(f'ph_agg: {ph_agg.shape[0]:,} loans')
print(ph_agg[['on_time_rate','total_days_delinquent']].describe())

ph_agg: 180,000 loans
       on_time_rate  total_days_delinquent
count   180000.0000            180000.0000
mean         0.9276                 0.5402
std          0.1899                 1.6471
min          0.0000                 0.0000
25%          1.0000                 0.0000
50%          1.0000                 0.0000
75%          1.0000                 0.0000
max          1.0000                24.0000


### 2.2 Merge tables + encode + impute

In [72]:
df = loans.merge(borrowers, on='borrower_id', how='left')
df = df.merge(ph_agg, on='loan_id', how='left')

# Loans with no payment record: flag = 0, fill metrics with conservative defaults
df['has_payment_history']    = df['has_payment_history'].fillna(0).astype(int)
df['on_time_rate']           = df['on_time_rate'].fillna(df['on_time_rate'].median())
df['total_days_delinquent']  = df['total_days_delinquent'].fillna(0.0)

# Clip negative principals (data artifact)
df['principal'] = df['principal'].clip(lower=0)

# Encode categoricals — normalize case first (raw data has auto/Auto/AUTO variants)
df['loan_type_clean'] = df['loan_type'].str.strip().str.lower().fillna('unknown')
le_lt = LabelEncoder()
le_st = LabelEncoder()
df['loan_type_enc'] = le_lt.fit_transform(df['loan_type_clean'])
df['state_enc']     = le_st.fit_transform(df['state'].fillna('Unknown'))

# Median imputation for numeric cols with nulls
num_cols = [
    'credit_score', 'annual_income', 'employment_years',
    'debt_to_income_ratio', 'num_existing_loans', 'previous_defaults',
    'months_since_last_inquiry', 'principal', 'interest_rate',
    'term_months', 'monthly_payment', 'payment_to_income_ratio',
]
for c in num_cols:
    df[c] = df[c].fillna(df[c].median())

print(f'Merged df: {df.shape}')
print(f'Null count after imputation: {df[num_cols].isnull().sum().sum()}')

Merged df: (183596, 24)
Null count after imputation: 0


## 3 · Train Random Forest → `predicted_default_proba`

**Why re-train here instead of loading submission.csv?**  
`submission.csv` only stores the binary predicted flag (0/1), not the raw probability. 
We need the continuous probability to split loans into meaningful risk bands 
and compute priority scores.

We train on the labeled 126K train set and then call `predict_proba` on all 180K loans.

In [73]:
feature_cols = [
    'principal', 'interest_rate', 'term_months', 'monthly_payment',
    'payment_to_income_ratio', 'credit_score', 'annual_income',
    'employment_years', 'debt_to_income_ratio', 'num_existing_loans',
    'previous_defaults', 'months_since_last_inquiry',
    'loan_type_enc', 'state_enc',
    'on_time_rate', 'total_days_delinquent', 'has_payment_history',
]

# Training set: labeled loans only
df_train = df.merge(train_labels[['loan_id', 'default_flag']], on='loan_id', how='inner')
X_tr = df_train[feature_cols]
y_tr = df_train['default_flag']

rf = RandomForestClassifier(
    n_estimators=100, max_depth=12, random_state=42, n_jobs=-1
)
rf.fit(X_tr, y_tr)

# Predict proba for ALL 180K loans
df['predicted_default_proba'] = rf.predict_proba(df[feature_cols])[:, 1]

print(f'RF trained on {len(df_train):,} labeled loans')
print('\npredicted_default_proba distribution:')
print(df['predicted_default_proba'].describe())

RF trained on 128,528 labeled loans

predicted_default_proba distribution:
count   183596.0000
mean         0.1795
std          0.3467
min          0.0015
25%          0.0119
50%          0.0266
75%          0.0686
max          1.0000
Name: predicted_default_proba, dtype: float64


---
## Step 1 · Risk Band Assignment

| Band | Threshold | Interpretation |
|---|---|---|
| **Low** | proba < 0.10 | Routine monitoring |
| **Medium** | 0.10 – 0.30 | Watch list |
| **High** | 0.30 – 0.60 | Active oversight |
| **Critical** | ≥ 0.60 | Immediate intervention |

In [74]:
df['risk_band'] = pd.cut(
    df['predicted_default_proba'],
    bins=BAND_BINS,
    labels=BAND_LABELS,
    right=False,
)

band_counts = df['risk_band'].value_counts().reindex(BAND_LABELS)
band_pct    = (band_counts / len(df) * 100).round(2)

print('=== Risk Band Distribution ===')
for band, cnt, pct in zip(BAND_LABELS, band_counts, band_pct):
    bar = '█' * int(pct)
    print(f'  {band:<10} {cnt:>7,}  ({pct:5.1f}%)  {bar}')

=== Risk Band Distribution ===
  Low        146,879  ( 80.0%)  ████████████████████████████████████████████████████████████████████████████████
  Medium       8,438  (  4.6%)  ████
  High           694  (  0.4%)  
  Critical    27,585  ( 15.0%)  ███████████████


---
## Step 2 · Risk Profile — per-band summary statistics

For each band: loan count, avg probability, average principal, avg on_time_rate, 
avg delinquent days, and loan_type breakdown.

### 2.1 Core numeric profile

In [75]:
# Core numeric profile ────────────────────────────────────────────────────────
profile_num = (
    df.groupby('risk_band', observed=True).agg(
        loan_count             = ('loan_id',                   'count'),
        avg_default_proba      = ('predicted_default_proba',   'mean'),
        avg_principal          = ('principal',                  'mean'),
        total_principal        = ('principal',                  'sum'),
        avg_on_time_rate       = ('on_time_rate',               'mean'),
        avg_days_delinquent    = ('total_days_delinquent',      'mean'),
        avg_credit_score       = ('credit_score',               'mean'),
        avg_dti                = ('debt_to_income_ratio',       'mean'),
        pct_has_payment_hist   = ('has_payment_history',        'mean'),
    )
    .reindex(BAND_LABELS)
    .round(4)
)
profile_num['loan_pct'] = (profile_num['loan_count'] / len(df) * 100).round(2)
profile_num['total_principal'] = profile_num['total_principal'].astype(int)

print('=== Numeric Profile by Risk Band ===')
profile_num

=== Numeric Profile by Risk Band ===


,loan_count,avg_default_proba,avg_principal,total_principal,avg_on_time_rate,avg_days_delinquent,avg_credit_score,avg_dti,pct_has_payment_hist,loan_pct
risk_band,,,,,,,,,,
Low,146879,0.0264,85016.5132,12487140449,1.0000,0.0000,595.5535,35.1324,1.0000,80.0000
Medium,8438,0.1500,64473.7470,544029476,1.0000,0.0000,595.6901,35.2523,1.0000,4.6000
High,694,0.3951,61984.1472,43016998,1.0000,0.0000,592.4948,35.9321,1.0000,0.3800
Critical,27585,0.9981,74194.9214,2046666907,0.5185,3.5907,595.9674,35.1584,1.0000,15.0200


### 2.2 Loan-type composition per band

In [76]:
# Loan-type % within each band ───────────────────────────────────────────────
lt_cross = (
    df.groupby(['risk_band', 'loan_type_clean'], observed=True)['loan_id']
    .count()
    .reset_index(name='count')
)
lt_cross['band_total'] = lt_cross.groupby('risk_band')['count'].transform('sum')
lt_cross['pct']        = (lt_cross['count'] / lt_cross['band_total'] * 100).round(2)
lt_cross = lt_cross.drop(columns='band_total')

# Pivot for readability
lt_pivot = (
    lt_cross.pivot(index='risk_band', columns='loan_type_clean', values='pct')
    .reindex(BAND_LABELS)
    .fillna(0)
    .round(2)
)
print('=== Loan Type % Composition by Risk Band ===')
lt_pivot

=== Loan Type % Composition by Risk Band ===


/var/folders/24/nygh9dm53l323tt1pngq7s5m0000gn/T/ipykernel_49620/1752695014.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  lt_cross['band_total'] = lt_cross.groupby('risk_band')['count'].transform('sum')


loan_type_clean,auto,credit_line,home_equity,personal,small_business,student,unknown
risk_band,,,,,,,
Low,6.1300,43.9600,41.3600,0.1300,7.4100,0.1900,0.8300
Medium,6.8700,43.5900,40.1400,0.1800,7.9400,0.1400,1.1400
High,7.7800,43.6600,39.3400,0.0000,7.2000,0.1400,1.8700
Critical,6.2100,44.1400,41.3400,0.1100,7.1900,0.1700,0.8300


### 2.3 Combine into single risk_profile DataFrame

In [77]:
risk_profile = profile_num.reset_index().rename(columns={'risk_band':'band'})

# Append a compact loan_type string for easy reading
def top_types(band_name, n=3):
    sub = lt_cross[lt_cross['risk_band'] == band_name].nlargest(n, 'pct')
    return ', '.join(f'{r.loan_type_clean} {r.pct:.0f}%' for _, r in sub.iterrows())

risk_profile['top_loan_types'] = risk_profile['band'].apply(top_types)

# Reorder columns for the output CSV
out_cols = [
    'band', 'loan_count', 'loan_pct',
    'avg_default_proba', 'avg_principal', 'total_principal',
    'avg_on_time_rate', 'avg_days_delinquent',
    'avg_credit_score', 'avg_dti',
    'pct_has_payment_hist', 'top_loan_types',
]
risk_profile = risk_profile[out_cols]

print('risk_profile preview:')
risk_profile

risk_profile preview:


,band,loan_count,loan_pct,avg_default_proba,avg_principal,total_principal,avg_on_time_rate,avg_days_delinquent,avg_credit_score,avg_dti,pct_has_payment_hist,top_loan_types
0,Low,146879,80.0000,0.0264,85016.5132,12487140449,1.0000,0.0000,595.5535,35.1324,1.0000,"credit_line 44%, home_equity 41%, small_busine..."
1,Medium,8438,4.6000,0.1500,64473.7470,544029476,1.0000,0.0000,595.6901,35.2523,1.0000,"credit_line 44%, home_equity 40%, small_busine..."
2,High,694,0.3800,0.3951,61984.1472,43016998,1.0000,0.0000,592.4948,35.9321,1.0000,"credit_line 44%, home_equity 39%, auto 8%"
3,Critical,27585,15.0200,0.9981,74194.9214,2046666907,0.5185,3.5907,595.9674,35.1584,1.0000,"credit_line 44%, home_equity 41%, small_busine..."


### 2.4 Export `risk_profile.csv`

In [78]:
risk_profile.to_csv(DATA_DIR / 'risk_profile.csv', index=False)
print(f'Saved risk_profile.csv — {len(risk_profile)} rows (one per band)')

Saved risk_profile.csv — 4 rows (one per band)


---
## ✓ Steps 1 & 2 Complete

`df` now carries `predicted_default_proba` and `risk_band` for all 180K loans.  
`risk_profile.csv` is saved.  

**Steps 3 & 4** (Early Warning + Priority Score) follow in the next section.

In [79]:
print(f'df shape: {df.shape}')
print(f'Columns added: predicted_default_proba, risk_band')
print()
print('Risk band counts:')
print(df['risk_band'].value_counts().reindex(BAND_LABELS))

df shape: (183596, 26)
Columns added: predicted_default_proba, risk_band

Risk band counts:
risk_band
Low         146879
Medium        8438
High           694
Critical     27585
Name: count, dtype: int64


---
## Step 3 · Early Warning — Deteriorating Loans

### 3.0 Why Low/Medium + `is_deteriorating` is structurally empty

Before running the analysis it's worth understanding the **data mechanics** so 
the empty result below is treated as an insight, not a bug:

| Chain | Consequence |
|---|---|
| `is_deteriorating = 1` | `on_time_rate_cumul` dropped > 10pp in this month |
| Drop > 10pp | At least one `days_delinquent > 0` payment exists |
| Any delinquency | Aggregated `on_time_rate` < 1.0 as a model feature |
| `on_time_rate` < 1.0 | RF assigns `predicted_default_proba ≈ 1.0` → **Critical** |

**Conclusion**: The risk band is a *lagging* indicator. By the time `is_deteriorating` 
fires, the loan has already crossed into Critical. Low and Medium bands are, by 
construction, composed entirely of loans with a perfect payment record.

This is itself a key interview insight: *"Payment behavior dominates every other 
feature. The model has essentially learned a single decision boundary — 
did this borrower ever miss a payment?"*

The two analyses below provide **actionable early warning signals** that work 
within this constraint.

### 3.1 Rebuild loan-level monthly snapshot

In [80]:
# Rebuild the expanding-window snapshot (same logic as Phase 0)
ph_ew = payment_history.sort_values(['loan_id','month_number']).copy()
ph_ew['days_delinquent_filled'] = ph_ew['days_delinquent'].fillna(0.0)
ph_ew['on_time_flag']           = (ph_ew['days_delinquent_filled'] == 0).astype(int)

grp_ew = ph_ew.groupby('loan_id', sort=False)
ph_ew['on_time_count_cumul']    = grp_ew['on_time_flag'].cumsum()
ph_ew['months_observed']        = grp_ew['on_time_flag'].cumcount() + 1
ph_ew['total_delinquent_cumul'] = grp_ew['days_delinquent_filled'].cumsum()
ph_ew['on_time_rate_cumul']     = ph_ew['on_time_count_cumul'] / ph_ew['months_observed']
ph_ew['on_time_rate_mom_change']= ph_ew.groupby('loan_id')['on_time_rate_cumul'].diff()
ph_ew['is_deteriorating']       = (
    ph_ew['on_time_rate_mom_change'] < DETERIORATION_THRESHOLD
).astype(int)

snap_ew = ph_ew[['loan_id','month_number','on_time_rate_cumul',
                  'total_delinquent_cumul','on_time_rate_mom_change',
                  'is_deteriorating']].copy()

# Attach risk band + loan attributes — include loan_type_clean & state
# so they flow through to priority_df without a second merge
band_lookup = df[['loan_id','risk_band','predicted_default_proba',
                   'principal','loan_type_clean','state']].copy()
snap_ew = snap_ew.merge(band_lookup, on='loan_id', how='left')

print(f'snap_ew shape: {snap_ew.shape}')
print('snap_ew columns:', snap_ew.columns.tolist())
print('risk_band x is_deteriorating cross-tab (any month):')
print(pd.crosstab(snap_ew['risk_band'], snap_ew['is_deteriorating'],
                  rownames=['risk_band'], colnames=['is_deteriorating']))


snap_ew shape: (640096, 11)
snap_ew columns: ['loan_id', 'month_number', 'on_time_rate_cumul', 'total_delinquent_cumul', 'on_time_rate_mom_change', 'is_deteriorating', 'risk_band', 'predicted_default_proba', 'principal', 'loan_type_clean', 'state']
risk_band x is_deteriorating cross-tab (any month):
is_deteriorating       0      1
risk_band                      
Low               529871      0
Medium             23404      0
High                1677      0
Critical           51292  33852


### 3.2 Confirm: Low/Medium + `is_deteriorating=1` cross-section is empty

The cross-tab above confirms the structural constraint. The text below explains it for 
any reviewer reading this notebook.

In [81]:
hidden = snap_ew.loc[
    snap_ew['risk_band'].isin(['Low','Medium']) &
    (snap_ew['is_deteriorating'] == 1)
]
print(f'Low/Medium + is_deteriorating=1 rows: {len(hidden)}')
print()
print('Explanation:')
print('  is_deteriorating=1 requires on_time_rate_cumul < 1.0 (a delinquent payment).')
print('  Any delinquency makes the RF predict proba ≈ 1.0 → Critical band.')
print('  Risk band is a lagging indicator: the loan is already Critical')
print('  by the time the deterioration signal fires.')
print()
print('  Interview framing: "Payment behavior is so dominant that the model')
print('  has essentially a single decision boundary — has this borrower ever')
print('  missed a payment? Any miss → Critical."')

Low/Medium + is_deteriorating=1 rows: 0

Explanation:
  is_deteriorating=1 requires on_time_rate_cumul < 1.0 (a delinquent payment).
  Any delinquency makes the RF predict proba ≈ 1.0 → Critical band.
  Risk band is a lagging indicator: the loan is already Critical
  by the time the deterioration signal fires.

  Interview framing: "Payment behavior is so dominant that the model
  has essentially a single decision boundary — has this borrower ever
  missed a payment? Any miss → Critical."


### 3.3 Early Warning Signal A — 'Freshly Critical' loans

Critical loans that were **fully clean just one month ago** (`on_time_rate_cumul = 1.0` 
at `month_number − 1`, now `< 1.0`). These are loans that a real-time monitoring 
system should have flagged the moment they first missed a payment — they represent 
the intervention window before the loan deteriorates further.

**Use case**: Outreach in the first 30 days after a missed payment has the 
highest recovery rate.

In [82]:
# For each Critical loan, find the first month where on_time_rate_cumul < 1.0
crit_ids = band_lookup[band_lookup['risk_band']=='Critical']['loan_id']
snap_crit = snap_ew[snap_ew['loan_id'].isin(crit_ids)].sort_values(
    ['loan_id','month_number']
)

# First delinquent month per Critical loan
first_bad = (
    snap_crit[snap_crit['on_time_rate_cumul'] < 1.0]
    .groupby('loan_id')['month_number'].min()
    .reset_index(name='first_delinquent_month')
)

print('=== Critical loans by first delinquent month ===')
timing = first_bad['first_delinquent_month'].value_counts().sort_index()
for month, cnt in timing.items():
    bar = '█' * (cnt // 200)
    print(f'  Month {month}: {cnt:>6,} loans  {bar}')

print(f'\nTotal Critical loans with payment record: {len(first_bad):,}')
pct_month2 = timing.get(2, 0) / len(first_bad) * 100
print(f'Cliff at Month 2: {timing.get(2,0):,} loans ({pct_month2:.1f}%) went bad')
print(f'→ These were indistinguishable from Low-risk loans at Month 1.')

=== Critical loans by first delinquent month ===
  Month 1:  3,229 loans  ████████████████
  Month 2: 12,454 loans  ██████████████████████████████████████████████████████████████
  Month 3:  9,202 loans  ██████████████████████████████████████████████
  Month 4:  2,048 loans  ██████████
  Month 5:     75 loans  

Total Critical loans with payment record: 27,008
Cliff at Month 2: 12,454 loans (46.1%) went bad
→ These were indistinguishable from Low-risk loans at Month 1.


In [83]:
# 'Freshly Critical' = Critical loans whose first bad month was the LATEST month
# observed in the snapshot (still very early in deterioration trajectory)
latest_month_per_loan = snap_ew.groupby('loan_id')['month_number'].max().reset_index(
    name='last_month'
)
freshly_critical = (
    first_bad
    .merge(latest_month_per_loan, on='loan_id')
    .query('first_delinquent_month == last_month')   # went bad ONLY in their final month
    .merge(band_lookup, on='loan_id')
    .merge(snap_ew.groupby('loan_id').last().reset_index(
        )[['loan_id','on_time_rate_cumul','total_delinquent_cumul','on_time_rate_mom_change']],
        on='loan_id'
    )
    .sort_values('principal', ascending=False)
    .reset_index(drop=True)
)
freshly_critical.index += 1

print(f'Freshly Critical loans (went bad only in last observed month): {len(freshly_critical):,}')
print('\nTop 10 by principal:')
freshly_critical[['loan_id','first_delinquent_month','last_month',
                   'on_time_rate_cumul','total_delinquent_cumul',
                   'principal','predicted_default_proba']].head(10)

Freshly Critical loans (went bad only in last observed month): 8,121

Top 10 by principal:


,loan_id,first_delinquent_month,last_month,on_time_rate_cumul,total_delinquent_cumul,principal,predicted_default_proba
1,L7059-16180,2,2,0.5000,1.0000,242229.3904,1.0000
2,L53867-87904,3,3,0.6667,1.0000,242229.3904,1.0000
3,L4527-86740,2,2,0.5000,2.0000,242229.3904,1.0000
4,L79756-355478,3,3,0.6667,1.0000,242229.3904,1.0000
5,L86551-99552,3,3,0.6667,1.0000,242229.3904,1.0000
6,L69982-991329,4,4,0.7500,1.0000,242229.3904,1.0000
7,L89103-969028,2,2,0.5000,2.0000,242229.3904,0.9900
8,L31171-56092,4,4,0.7500,2.0000,242229.3904,1.0000
9,L88978-843052,2,2,0.5000,10.0000,242229.3904,1.0000
10,L27162-174493,4,4,0.7500,2.0000,242229.3904,1.0000


### 3.4 Early Warning Signal B — Elevated watch list within Low/Medium

Within the Low and Medium bands, not all loans are equal. Loans in the **top quartile 
of `predicted_default_proba` within their band** carry the highest structural risk 
while still having a clean payment record. These are the next most likely to tip 
into Critical territory.

| Band | Threshold | Count in watch list |
|---|---|---|
| Low (< 0.10) | top 25% within band, i.e., proba ≥ p75 | ~37K |
| Medium (0.10–0.30) | top 25% within band | ~2K |

In [84]:
low_med = df[df['risk_band'].isin(['Low','Medium'])].copy()

# Top-quartile within each band
low_med['proba_pctile_in_band'] = (
    low_med.groupby('risk_band', observed=True)['predicted_default_proba']
    .rank(pct=True)
)
watch_list = low_med[low_med['proba_pctile_in_band'] >= 0.75].copy()

print('=== Elevated Watch List (top-quartile within Low/Medium) ===')
wl_summary = watch_list.groupby('risk_band', observed=True).agg(
    count               = ('loan_id',                'count'),
    proba_min           = ('predicted_default_proba', 'min'),
    proba_max           = ('predicted_default_proba', 'max'),
    avg_principal       = ('principal',               'mean'),
).round(4)
print(wl_summary)

print(f'\nTotal watch-list loans: {len(watch_list):,}')
print(f'These carry clean payment records but have the highest structural risk')
print(f'within their cohort — first candidates for pre-emptive outreach.')

=== Elevated Watch List (top-quartile within Low/Medium) ===
           count  proba_min  proba_max  avg_principal
risk_band                                            
Low        36720     0.0365     0.1000     74338.8334
Medium      2110     0.1738     0.2997     62499.3894

Total watch-list loans: 38,830
These carry clean payment records but have the highest structural risk
within their cohort — first candidates for pre-emptive outreach.


In [85]:
# Export watch list
watch_list[['loan_id','risk_band','predicted_default_proba',
            'proba_pctile_in_band','principal',
            'loan_type_clean','state']]\
    .sort_values(['risk_band','predicted_default_proba'], ascending=[True,False])\
    .reset_index(drop=True)\
    .to_csv(DATA_DIR / 'early_warning_watchlist.csv', index=False)
print(f'Saved early_warning_watchlist.csv — {len(watch_list):,} loans')

Saved early_warning_watchlist.csv — 38,830 loans


### 3.5 Step 3 Summary

| Signal | Count | Action |
|---|---|---|
| **Freshly Critical** | ~3–4K | Immediate outreach — just crossed threshold |
| **Elevated Watch List** (Low/Medium top-quartile) | ~39K | Pre-emptive monitoring |

> **Interview talking point**: *"Because payment behavior dominates, traditional 
> early warning must work on leading indicators — structural stress signals — 
> rather than waiting for the first missed payment. By the time `is_deteriorating` 
> fires, the loan has already crossed into Critical."*

---
## Step 4 · Intervention Priority Score

Combines three risk dimensions into a single score to rank which loans 
the collections team should call first:

$$\text{priority\_score} = \text{predicted\_default\_proba} \times \text{principal} \times (1 + \text{is\_deteriorating})$$

| Component | Logic |
|---|---|
| `predicted_default_proba` | how likely is this loan to default? |
| `principal` | how much money is at risk? |
| `1 + is_deteriorating` | doubles score for loans actively worsening this month |

### 4.1 Compute priority score — latest snapshot row per loan

In [86]:
# Use the most recent monthly state per loan
# NOTE: snap_ew already carries predicted_default_proba, principal, risk_band
# from the Step 3 merge — no second join needed.
latest_snap = (
    snap_ew.sort_values(['loan_id','month_number'])
    .groupby('loan_id').last().reset_index()
)

priority_df = latest_snap.copy()   # proba / principal / risk_band already present

priority_df['priority_score'] = (
    priority_df['predicted_default_proba']
    * priority_df['principal']
    * (1 + priority_df['is_deteriorating'])
)

print('priority_score statistics:')
print(priority_df['priority_score'].describe())
print()
print('Non-zero priority loans:', (priority_df['priority_score'] > 0).sum())

priority_score statistics:
count   180000.0000
mean     20402.8823
std      46103.9958
min          0.0000
25%       1039.3645
50%       2130.7322
75%       4984.6408
max     484458.7808
Name: priority_score, dtype: float64

Non-zero priority loans: 179507


### 4.2 Top 50 priority loans

In [87]:
# risk_band, loan_type_clean, state are already in priority_df
# (inherited from snap_ew → band_lookup). No re-merge needed.

top50 = (
    priority_df
    .nlargest(50, 'priority_score')
    [['loan_id','risk_band','predicted_default_proba','principal',
      'on_time_rate_cumul','total_delinquent_cumul',
      'is_deteriorating','priority_score',
      'loan_type_clean','state']]
    .reset_index(drop=True)
)
top50.index += 1
top50.index.name = 'rank'

# Format for display
top50_display = top50.copy()
top50_display['priority_score']          = top50_display['priority_score'].round(0).astype(int)
top50_display['principal']               = top50_display['principal'].round(0).astype(int)
top50_display['predicted_default_proba'] = top50_display['predicted_default_proba'].round(4)
top50_display['on_time_rate_cumul']      = top50_display['on_time_rate_cumul'].round(4)

print('=== Top 50 Intervention Priority Loans ===')
print(top50_display[['loan_id','risk_band','predicted_default_proba',
                      'principal','is_deteriorating',
                      'priority_score','loan_type_clean','state']].to_string())


=== Top 50 Intervention Priority Loans ===
            loan_id risk_band  predicted_default_proba  principal  is_deteriorating  priority_score loan_type_clean state
rank                                                                                                                     
1     L14492-543328  Critical                   1.0000     242229                 1          484459     credit_line    AL
2     L15625-229307  Critical                   1.0000     242229                 1          484459     home_equity    GU
3     L17211-139400  Critical                   1.0000     242229                 1          484459     home_equity    RI
4      L17233-31281  Critical                   1.0000     242229                 1          484459     credit_line    CT
5     L27162-174493  Critical                   1.0000     242229                 1          484459     credit_line    NJ
6     L29842-257525  Critical                   1.0000     242229                 1          484459    

### 4.3 Priority score distribution by risk band

In [88]:
ps_by_band = (
    priority_df.groupby('risk_band', observed=True)['priority_score']
    .agg(['count','sum','mean','median','max'])
    .reindex(BAND_LABELS)
    .round(2)
)
ps_by_band.columns = ['loan_count','total_score','avg_score','median_score','max_score']
ps_by_band['total_score'] = ps_by_band['total_score'].round(0).astype(int)
ps_by_band['max_score']   = ps_by_band['max_score'].round(0).astype(int)

print('=== Priority Score Distribution by Risk Band ===')
ps_by_band

=== Priority Score Distribution by Risk Band ===


,loan_count,total_score,avg_score,median_score,max_score
risk_band,,,,,
Low,144020,299462399,2079.3100,1639.4100,23457
Medium,8274,79451103,9602.5000,8795.1500,59255
High,671,16405794,24449.7700,22744.0900,114478
Critical,27035,3277199526,121220.6200,123778.1400,484459


### 4.4 Export `priority_loans.csv`

In [89]:
top50.to_csv(DATA_DIR / 'priority_loans.csv')
print(f'Saved priority_loans.csv — {len(top50)} loans')
print()
print('Columns:')
for c in top50.columns:
    print(f'  {c}')

Saved priority_loans.csv — 50 loans

Columns:
  loan_id
  risk_band
  predicted_default_proba
  principal
  on_time_rate_cumul
  total_delinquent_cumul
  is_deteriorating
  priority_score
  loan_type_clean
  state


---
## ✓ Phase 2 Complete

| File | Contents |
|---|---|
| `risk_profile.csv` | 4 rows — per-band summary statistics |
| `early_warning_watchlist.csv` | ~39K rows — Low/Medium loans in top-quartile proba |
| `priority_loans.csv` | Top 50 — ranked by `proba × principal × (1 + deteriorating)` |

### Key Takeaways

1. **Risk is binary, not a spectrum** — payment behavior creates a hard cliff 
   between Critical (15%, on_time=0.52) and everyone else (85%, on_time=1.0).
2. **Traditional metrics don't differentiate** — credit score, DTI, employment 
   are near-identical across all four bands. Payment history is the sole signal.
3. **Early warning must be proactive** — by the time deterioration is observable, 
   the loan is already Critical. The watch list uses *relative standing within a 
   band* as a leading indicator.
4. **Month 2 is the critical intervention window** — the largest single-month 
   cliff in Critical loans is the transition from Month 1 (clean) to Month 2 
   (first missed payment).

In [90]:
print('Phase 2 output files:')
for fname in ['risk_profile.csv','early_warning_watchlist.csv','priority_loans.csv']:
    p = DATA_DIR / fname
    if p.exists():
        kb = p.stat().st_size / 1024
        print(f'  ✓ {fname:<35}  {kb:,.0f} KB')
    else:
        print(f'  ✗ {fname}  NOT FOUND')

Phase 2 output files:
  ✓ risk_profile.csv                     1 KB
  ✓ early_warning_watchlist.csv          3,055 KB
  ✓ priority_loans.csv                   5 KB
